# Practical 6: Fine-Tuning & Layer Freezing Strategies on MobileNet
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit III (Transfer Learning & Fine-Tuning) | **CO:** CO3  
**Problem Statement Context:** Mobile Edge Deployment for In-Field Crop Diagnostics  

---

## 1. Transfer Learning Spectrum
When adapting a pre-trained network to agricultural imagery:
1. **Strategy 1: Frozen Backbone (Feature Extraction):** Fast training, zero risk of catastrophic forgetting, but lower peak accuracy.
2. **Strategy 2: Partial Unfreezing (Top Blocks):** Retains generic low-level filters (edges, textures) in early layers while adapting high-level semantic representations (leaf shape, lesion morphology) in upper layers.
3. **Strategy 3: Full Fine-Tuning with Differential Learning Rates:**
   $$\eta_{backbone} = 10^{-5}, \quad \eta_{head} = 10^{-3}$$
   Allows subtle backbone parameter adjustments without destroying pre-trained weights.

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.dataset import PlantDiseaseClassificationDataset
from src.transforms import get_classification_transforms
from src.models_cnn import build_transfer_classifier, unfreeze_top_layers, get_differential_lr_params

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"[✓] Active Device: {device}")

## 2. Benchmark Setup: Freezing vs. Partial Unfreezing

In [ ]:
data_dir = REPO_ROOT / "data" / "samples" / "classification" / "lab_plantvillage"
transforms = get_classification_transforms(image_size=(224, 224), is_training=True)
dataset = PlantDiseaseClassificationDataset(root_dir=data_dir, transform=transforms)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# 1. Strategy A: Frozen
model_frozen = build_transfer_classifier("mobilenet_v2", num_classes=len(dataset.classes), freeze_backbone=True).to(device)
opt_frozen = optim.Adam([p for p in model_frozen.parameters() if p.requires_grad], lr=1e-3)

# 2. Strategy B: Partial Unfreezing (Unfreeze top layers)
model_unfrozen = build_transfer_classifier("mobilenet_v2", num_classes=len(dataset.classes), freeze_backbone=True).to(device)
unfreeze_top_layers(model_unfrozen, num_layers=4)
opt_unfrozen = optim.Adam(get_differential_lr_params(model_unfrozen, base_lr=1e-5, head_lr_mult=100.0))

print("[✓] Both fine-tuning configurations initialized.")

## 3. Compare Convergence over 4 Epochs

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_epochs(model, optimizer, loader, epochs=4):
    history = []
    for ep in range(epochs):
        model.train()
        loss_sum = 0.0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()
        history.append(loss_sum / len(loader))
    return history

hist_frozen = train_epochs(model_frozen, opt_frozen, dataloader)
hist_unfrozen = train_epochs(model_unfrozen, opt_unfrozen, dataloader)

plt.figure(figsize=(8, 5))
plt.plot(hist_frozen, label="Strategy A: Fully Frozen Backbone", marker='o')
plt.plot(hist_unfrozen, label="Strategy B: Partial Fine-Tuning (Top 4 Layers)", marker='s')
plt.title("Fine-Tuning Convergence Comparison on MobileNet")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Engineering Recommendations
- When target dataset is **small and domain-similar** (e.g., standard leaves), freeze the backbone to avoid overfitting.
- When target images exhibit **domain shift** (e.g., lab PlantVillage to noisy field PlantDoc), unfreeze the top residual blocks with differential learning rates to adapt high-level disease pattern recognition.